# 🧪 Thí nghiệm E3 + E4 — đánh giá phần ML (notebook của Quyên)

**E3** — đánh giá **CafeBERT fine-tuned** (`BaoNhan/cafebert-UITVSMEC`) trên tập **test của UIT-VSMEC**:
accuracy, macro-F1, weighted-F1, per-class + confusion matrix.

**E4** — **Qwen3-4B zero-shot** (không fine-tune) trên cùng tập test: so sánh LLM tổng quát
vs model fine-tuned chuyên biệt → bảng so sánh cho báo cáo.

Nguồn dataset: [ura-hcmut/UIT-VSMEC (HuggingFace)](https://huggingface.co/datasets/ura-hcmut/UIT-VSMEC)
— 6.927 câu Facebook, 7 nhãn (UIT-VSMEC, PACLING 2019).

⚠️ Chạy trên **Colab T4 GPU** (có thể chạy ngay trong runtime đang mở model server).
Tham chiếu điểm mục tiêu: CafeBERT ~0.627 macro-F1 trên UIT-VSMEC (model card).

In [ ]:
# Cấu hình chung
import os, json

# Model server đang chạy (dùng cho E4 — không tải thêm LLM vào VRAM)
os.environ["SERV_URL"] = "https://pang-unreeling-shriek.ngrok-free.dev"
try:
    from google.colab import userdata
    os.environ["SERV_TOKEN"] = userdata.get("MODEL_SERVER_TOKEN")
except Exception as e:
    print("⚠️ Lấy secret:", e)

import torch
print("GPU:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")
import datasets, transformers
print("datasets:", datasets.__version__, "| transformers:", transformers.__version__)

## 1. Nạp UIT-VSMEC (train/validation/test)

In [ ]:
from collections import Counter

ds = datasets.load_dataset("ura-hcmut/UIT-VSMEC")
print(ds)

def pick(name):
    for alias in (name, name.lower()):
        if alias in ds: return ds[alias]
    return None

test_ds = pick("test") or ds[list(ds.keys())[-1]]
print("test size:", len(test_ds))
first = test_ds[0]
for k, v in first.items(): print(f"  {k!r}: {str(v)[:80]!r}")

## 2. Tự động khớp cột text + cột nhãn ↔ 7 nhãn chuẩn của module emotion

Tên cột dataset chỉ có trên HF — script này khớp theo **giá trị** (an toàn hơn tên cột).

In [ ]:
LABELS = ["Anger", "Disgust", "Enjoyment", "Fear", "Other", "Sadness", "Surprise"]

def detect_cols(ds_split):
    """Trả (TEXT_COL, LAB_COL) bằng cách kiểm tra giá trị của vài dòng đầu."""
    sample = [ds_split[i] for i in range(min(20, len(ds_split)))]
    best_lab, best_text = None, None
    for col in sample[0].keys():
        vals = [row[col] for row in sample]
        strvals = [str(v).strip().lower() for v in vals]
        # nhãn: >=80% giá trị thuộc 7 nhãn chuẩn (lowercase) HOẶC số nguyên 0..6
        hit_lab = sum(v in [l.lower() for l in LABELS] for v in strvals)
        if best_lab is None or hit_lab > best_lab[1]:
            best_lab = (col, hit_lab)
        # text: chuỗi dài nhất trung bình, không phải cột nhãn
        if all(isinstance(v, str) for v in vals):
            avg = sum(len(v) for v in vals) / len(vals)
            if best_text is None or avg > best_text[1]:
                best_text = (col, avg)
    return best_text[0], best_lab[0]

TEXT_COL, LAB_COL = detect_cols(test_ds)
print("TEXT_COL:", TEXT_COL, "| LAB_COL:", LAB_COL)

# Map giá trị nhãn dataset -> nhãn chuẩn (lowercase khớp tên; số 0..6 khớp alphabet LABELS)
from collections import Counter
uniques = Counter(str(v).strip().lower() for v in test_ds[LAB_COL])
lower2label = {l.lower(): l for l in LABELS}
is_numeric = all(u.isdigit() for u in uniques)
val_map = {u: lower2label.get(u) for u in uniques}  # None nếu không khớp
if is_numeric:
    val_map = {str(i): LABELS[i] for i in range(7)}  # thứ tự alphabet = LABELS
unknown = [u for u, m in val_map.items() if m is None]
assert not unknown, f"Giá trị nhãn lạ, cần map tay: {unknown}"
print("map giá trị nhãn:", val_map)

## 3. E3 — CafeBERT fine-tuned trên tập test

In [ ]:
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer
from sklearn.metrics import accuracy_score, f1_score, classification_report

MODEL_NAME = "BaoNhan/cafebert-UITVSMEC"
tok = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=False)
mdl = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME).eval().cuda()

# id -> nhãn chuẩn (giống module emotion: tra cả key int và str, fallback theo vị trí)
cfg = mdl.config
assert cfg.num_labels == 7
id2l = {}
for i in range(7):
    raw = str(cfg.id2label.get(i, cfg.id2label.get(str(i), ""))).strip().title().split("_")[-1]
    id2l[i] = raw if raw in LABELS else LABELS[i]
print("id2label:", id2l)

texts = list(test_ds[TEXT_COL])
gold  = [val_map[str(v).strip().lower()] for v in test_ds[LAB_COL]]
from collections import Counter
print("tổng test:", len(gold), "| phân bố:", Counter(gold))

In [ ]:
# Batch inference toàn tập test (693 câu — vài phút trên T4)
BATCH = 64
@torch.no_grad()
def predict_batch(batch_texts):
    enc = tok(batch_texts, return_tensors="pt", padding=True, truncation=True, max_length=256).to("cuda")
    return mdl(**enc).logits.float().softmax(dim=-1)

probs_rows, preds = [], []
for i in range(0, len(texts), BATCH):
    p = predict_batch(texts[i:i+BATCH])
    probs_rows.extend(p.cpu().tolist())
    preds.extend([id2l[int(a)] for a in p.argmax(-1).cpu()])
    print(f"\r  {i + min(BATCH, len(texts)-i)}/{len(texts)}", end="")
print("\nXong inference.")

acc = accuracy_score(gold, preds)
f1m = f1_score(gold, preds, average="macro")
f1w = f1_score(gold, preds, average="weighted")
print(f"\n=== E3 CafeBERT fine-tuned ===\naccuracy   : {acc:.4f}\nmacro-F1   : {f1m:.4f}  (mục tiêu tham chiếu ~0.627)\nweighted-F1: {f1w:.4f}")
print(classification_report(gold, preds, labels=LABELS, digits=3))

E3 = {"accuracy": round(acc, 4), "macro_f1": round(f1m, 4), "weighted_f1": round(f1w, 4)}
with open("/content/E3_results.json", "w") as f: json.dump(E3, f)
print("Lưu /content/E3_results.json:", E3)

In [ ]:
# Confusion matrix (lưu PNG làm hình báo cáo) + F1 từng nhãn
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, classification_report

cm = confusion_matrix(gold, preds, labels=LABELS)
fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(cm, cmap="Blues")
ax.set_xticks(range(7)); ax.set_xticklabels(LABELS, rotation=45, ha="right")
ax.set_yticks(range(7)); ax.set_yticklabels(LABELS)
for r in range(7):
    for c in range(7):
        ax.text(c, r, int(cm[r][c]), ha="center",
                color="white" if cm[r][c] > cm.max()/2 else "black", fontsize=8)
ax.set_xlabel("Dự đoán"); ax.set_ylabel("Thực tế"); ax.set_title("E3 — CafeBERT trên UIT-VSMEC test")
fig.colorbar(im); fig.tight_layout(); fig.savefig("/content/E3_confusion.png", dpi=150)
print("Đã lưu /content/E3_confusion.png")

rep = classification_report(gold, preds, labels=LABELS, output_dict=True, zero_division=0)
print("F1 từng nhãn:", {l: round(rep[l]["f1-score"], 3) for l in LABELS})

## 4. E4 — Qwen3-4B zero-shot **v3** (raw LLM local, greedy deterministic)

**Lịch sử protocol (ablation cho báo cáo):**
| | Endpoint | Decoding | Kết quả | Đánh giá |
|---|---|---|---|---|
| v1 | server `/generate` | sampling 0.7 (không kiểm soát) | 0.2267 / macro-F1 0.0588 | toàn fallback → parse-fail |
| v2 | server `/generate` | như trên | parse 85% nhưng **148/150 = "Other"** | **confound**: endpoint bọc system-prompt chatbot ("Hệ thống phát hiện cảm xúc: Other") → model bị chi phối nhãn ngữ cảnh |
| **v3** | **Qwen tải trực tiếp (raw)** | **greedy (temperature=0), max_new_tokens=8** | ⟵ cell này | chuẩn evaluation: deterministic, không wrapper |

Cơ sở: Hands-On LLMs (O'Reilly 2024) tr.195 — zero-shot classification cần **constrain output**;
số liệu v2 chứng minh endpoint chatbot-wrapped là sai phương pháp cho probe bên ngoài.

In [ ]:
# ---- E4-v3: Qwen3-4B raw, greedy deterministic ----
import os, json, time, random

N_E4, SEED = 150, 42
random.seed(SEED)
idx4 = random.sample(range(len(texts)), min(N_E4, len(texts)))   # cùng index v1/v2 (INV1)

VERBALIZERS = {
    "Anger":     ["Anger", "Giận", "Tức giận"],
    "Disgust":   ["Disgust", "Ghê", "Ghê tởm", "Khó chịu"],
    "Enjoyment": ["Enjoyment", "Vui", "Vui vẻ", "Hài lòng"],
    "Fear":      ["Fear", "Sợ", "Lo sợ", "Hoảng"],
    "Other":     ["Other", "Khác"],
    "Sadness":   ["Sadness", "Buồn"],
    "Surprise":  ["Surprise", "Ngạc nhiên", "Bất ngờ"],
}
assert set(VERBALIZERS) == set(LABELS)

SYSTEM = ("Bạn là công cụ phân loại cảm xúc. Chọn DUY NHẤT nhãn phù hợp nhất với câu "
          "và trả lời DUY NHẤT tên nhãn tiếng Anh, không thêm chữ nào khác. "
          "Nhãn: Anger (Giận dữ), Disgust (Ghê tởm), Enjoyment (Vui vẻ), Fear (Sợ hãi), "
          "Other (Khác), Sadness (Buồn), Surprise (Ngạc nhiên).")

def parse_reply(reply):
    if "<tool_call>think" in reply:
        reply = reply.rsplit("</think>", 1)[-1].strip()  # phong chat-template Qwen3
    r = reply.lower()
    en = [l for l, vs in VERBALIZERS.items() if vs[0].lower() in r]
    if len(en) == 1: return en[0], True
    vi = [l for l, vs in VERBALIZERS.items() if any(v.lower() in r for v in vs[1:])]
    if vi: return max(vi, key=lambda l: max(len(v) for v in VERBALIZERS[l][1:] if v.lower() in r)), True
    return "Other", False        # INV3: fallback duy nhất

# Tải raw Qwen3-4B — fp16, VRAM: CafeBERT(0.5) + Qwen(~8) < 15GB T4
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
q_name = os.environ.get("LLM_MODEL_ID", "Qwen/Qwen3-4B")
qtok = AutoTokenizer.from_pretrained(q_name)
qmod = AutoModelForCausalLM.from_pretrained(q_name, torch_dtype=torch.float16, device_map="cuda", low_cpu_mem_usage=True).eval()
print("Đã tải:", q_name)

In [ ]:
# ---- Inference greedy (deterministic) + parse stats + metrics ----
import torch
from collections import Counter

@torch.no_grad()
def classify_raw(text):
    msgs = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": text}]
    try:   # Qwen3 bat/tat "thinking" nhu llm.py; template cu (2.5) khong nhan kwarg
        prompt = qtok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True,
                                          enable_thinking=False)
    except (TypeError, KeyError):
        prompt = qtok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    enc = qtok([prompt], return_tensors="pt").to("cuda")
    out = qmod.generate(**enc, max_new_tokens=8, do_sample=False,  # greedy = temperature 0
                        pad_token_id=qtok.eos_token_id)
    reply = qtok.decode(out[0][enc["input_ids"].shape[-1]:], skip_special_tokens=True).strip()
    label, ok = parse_reply(reply)
    return label, reply, ok

qpred3, repl, ofs, t0 = [], [], [], time.time()
for k, i in enumerate(idx4):
    lab, rep, ok = classify_raw(texts[i])
    qpred3.append(lab); repl.append(rep); ofs.append(ok)
    if (k + 1) % 25 == 0: print(f"  {k+1}/{len(idx4)} | {time.time()-t0:.0f}s")
print("parse_stats:", {"parse_ok": sum(ofs), "fallback_other": len(idx4)-sum(ofs), "n": len(idx4)})
print("phân bố dự đoán:", Counter(qpred3))
print("5 reply đầu:", *[repr(r)[:40] for r in repl[:5]], sep="\n  ")

from sklearn.metrics import accuracy_score, f1_score
qgold = [gold[i] for i in idx4]
g = lambda p: (accuracy_score(qgold, p), f1_score(qgold, p, average="macro", zero_division=0),
               f1_score(qgold, p, average="weighted", zero_division=0))
a3, m3, w3 = g(qpred3)
print(f"\n=== E4-v3 Qwen3-4B raw ZERO-SHOT greedy (n={len(idx4)}, seed={SEED}) ===")
print(f"accuracy   : {a3:.4f}\nmacro-F1   : {m3:.4f}\nweighted-F1: {w3:.4f}")
if sum(ofs) / len(idx4) < 0.70:
    print("❗ KILL1: parse_ok < 70% — báo cáo chỉ ghi 'thất bại protocol', KHÔNG dùng số.")

E4v3 = {"accuracy": round(a3, 4), "macro_f1": round(m3, 4), "weighted_f1": round(w3, 4),
        "n": len(idx4), "seed": SEED, "protocol": "raw+greedy+verbalizer-vi-en",
        "parse_ok": sum(ofs)}
with open("/content/E4_v3_results.json", "w") as f: json.dump(E4v3, f)
with open("/content/E4_v3_replies.json", "w") as f:
    json.dump([{"i": i, "text": texts[i], "gold": gold[i], "reply": r, "pred": p}
               for i, r, p in zip(idx4, repl, qpred3)], f, ensure_ascii=False)
print("Lưu E4_v3_results.json + E4_v3_replies.json")

In [ ]:
# ---- So sánh tổng: E3 vs v1/v2 (lịch sử, ablation) vs v3 vs majority ----
from sklearn.metrics import f1_score
MAJ = max(set(gold), key=gold.count)
maj_acc = sum(1 for i in idx4 if gold[i] == MAJ) / len(idx4)

import pandas as pd
rows = [{"Bản": "E3 CafeBERT fine-tuned", "accuracy": E3["accuracy"],
         "macro-F1": E3["macro_f1"], "weighted-F1": E3["weighted_f1"], "n": len(gold)}]
try:
    e1 = json.load(open("/content/E4_results.json"))
    rows.append({"Bản": "v1 server+sampling (ablation)", "accuracy": e1.get("accuracy"),
                 "macro-F1": e1.get("macro_f1"), "weighted-F1": e1.get("weighted_f1"), "n": e1.get("n")})
except Exception:
    print("(Khong co file E4_v1 — dung so v1 lich su 0.2267/0.0588 vao bang bao cao)")
rows.append({"Bản": "v2 server+sampling, parse VI+EN (confound persona)", "accuracy": 0.2333,
             "macro-F1": 0.0657, "weighted-F1": 0.1056, "n": 150})
rows.append({"Bản": "v3 raw + greedy (chuẩn)", "accuracy": E4v3["accuracy"],
             "macro-F1": E4v3["macro_f1"], "weighted-F1": E4v3["weighted_f1"], "n": E4v3["n"]})
rows.append({"Bản": f"Majority-baseline (always-{MAJ})", "accuracy": round(maj_acc, 4),
             "macro-F1": None, "weighted-F1": None, "n": len(idx4)})
print(pd.DataFrame(rows).to_string(index=False))
print("\nBài học báo cáo: protocol quyết định số đo — v1 parse-fail, v2 persona-confound,")
print("v3 là phép đo zero-shot hợp lệ (deterministic, no wrapper).")